# 🛰️ Treatment Gap Radar
### Does global antimicrobial R&D go where resistance is actually getting worse?
*Vivli AMR Data Challenge — AMR ID 00013367*

This notebook integrates **13 antimicrobial-resistance surveillance datasets** (Vivli) with the
**Global AMR R&D Hub** investment/pipeline data to build two indices and compare them:

- **Resistance Need Index (RNI)** — six indicators of resistance pressure per pathogen.
- **R&D Attention Index (RAI)** — investment, project count, and pipeline products per pathogen.

The **gap** (RNI − RAI) surfaces pathogens where resistance is high but R&D attention is low.

> *This publication or presentation is based on research using data from GSK, Innoviva Specialty
> Therapeutics, Johnson & Johnson, Paratek, Pfizer, Shionogi, Venatorx, Venus Remedies Limited,
> obtained through https://amr.vivli.org*

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # embed figures as HTML (CDN plotly.js)
from src import viz
from src.paths import PROCESSED_DIR

# Build outputs if missing (reuses isolates_long.parquet if already harmonized).
if not (PROCESSED_DIR / "gap.parquet").exists():
    from src import pipeline
    pipeline.main(skip_harmonize=(PROCESSED_DIR / "isolates_long.parquet").exists())

long_cols = ["source", "pathogen", "antibiotic", "sir", "year", "country_iso3"]
print("Outputs ready in", PROCESSED_DIR)

Outputs ready in C:\Users\Niyel\Downloads\Data Challenge_Treatment Gap Radar (1)-20260622T211028Z-3-001\Data Challenge_Treatment Gap Radar (1)\treatment_gap_radar\data_processed


## 1 · Data harmonization
All datasets are normalized into one long table (isolate × antibiotic × S/I/R).

In [2]:
import pyarrow.parquet as pq
pf = pq.ParquetFile(PROCESSED_DIR / "isolates_long.parquet")
print(f"Harmonized rows: {pf.metadata.num_rows:,}")
src_counts = pd.read_parquet(PROCESSED_DIR / "isolates_long.parquet", columns=["source"])["source"].value_counts()
src_counts

Harmonized rows: 17,896,683


source
ATLAS          15792032
KEYSTONE        1297735
SIDERO_WT        388018
GEARS            185271
DREAM_TB          71135
SOAR_207965       47364
INNOVIVA          45288
SOAR_201910       32430
SOAR_201818       29235
PLEA_I             7315
GASAR_III           494
PLEA_II             232
SPIDAAR             134
Name: count, dtype: Int64

## 2 · Resistance indicators → Resistance Need Index
Six indicators per pathogen, normalized 0–1 and combined.

In [3]:
rni = pd.read_parquet(PROCESSED_DIR / "rni.parquet")
rni[["who", "n_isolates", "prevalence", "mic_drift", "mdr", "geo_spread", "scarcity", "pediatric", "RNI"]].round(3)

,who,n_isolates,prevalence,mic_drift,mdr,geo_spread,scarcity,pediatric,RNI
pathogen,,,,,,,,,
Enterococcus faecium,high,21831,0.374,0.006,0.564,0.955,0.700,0.330,0.843
Acinetobacter baumannii,critical,56625,0.494,0.003,0.606,0.750,0.600,0.304,0.806
Mycobacterium tuberculosis,critical,5928,0.465,0.009,0.496,1.000,1.000,NaN,0.772
Providencia stuartii,,2861,0.307,0.003,0.715,0.596,0.750,0.275,0.752
Klebsiella pneumoniae,critical,128382,0.263,0.003,0.381,0.617,0.769,0.259,0.655
Enterobacter cloacae,critical,51121,0.261,0.004,0.394,0.628,0.583,0.254,0.636
Staphylococcus epidermidis,,17698,0.235,-0.005,0.452,0.500,0.500,0.226,0.520
Escherichia coli,critical,155739,0.166,0.001,0.305,0.293,0.538,0.143,0.437
Serratia marcescens,,33394,0.202,-0.001,0.330,0.276,0.333,0.205,0.437


In [4]:
viz.indicator_heatmap(rni)

## 3 · R&D Attention Index
Text-attributed investment, project count, and pipeline products per pathogen (Global AMR R&D Hub).

In [5]:
rai = pd.read_parquet(PROCESSED_DIR / "rai.parquet")
rai[["gram", "projects", "investment", "pipeline", "named_projects", "RAI"]].round(2)

,gram,projects,investment,pipeline,named_projects,RAI
pathogen,,,,,,
Escherichia coli,negative,8308.0,8.955862e+09,222,1653,0.91
Pseudomonas aeruginosa,negative,7608.0,9.110575e+09,219,1004,0.87
Klebsiella pneumoniae,negative,7016.0,8.170417e+09,206,486,0.77
Acinetobacter baumannii,negative,6885.0,7.944071e+09,205,404,0.75
Salmonella enterica,negative,7243.0,7.811110e+09,191,798,0.74
Neisseria gonorrhoeae,negative,6669.0,7.766618e+09,200,258,0.72
Haemophilus influenzae,negative,6428.0,7.193725e+09,183,81,0.64
Enterobacter cloacae,negative,6458.0,7.232701e+09,181,92,0.64
Serratia marcescens,negative,6357.0,7.118548e+09,179,22,0.62


## 4 · The Treatment Gap
RNI (x) vs RAI (y). The **red quadrant** = high resistance need, low R&D attention — the priority gaps.

In [6]:
viz.gap_quadrant()

In [7]:
gap = pd.read_parquet(PROCESSED_DIR / "gap.parquet")
print("PRIORITY TREATMENT GAPS (high need / low attention):")
gap[gap["quadrant"].str.startswith("PRIORITY")].sort_values("gap_score", ascending=False)[
    ["who", "n_isolates", "RNI", "RAI", "gap_score"]].round(3)

PRIORITY TREATMENT GAPS (high need / low attention):


,who,n_isolates,RNI,RAI,gap_score
pathogen,,,,,
Enterococcus faecium,high,21831,0.843,0.140,0.704
Staphylococcus epidermidis,,17698,0.520,0.001,0.519
Mycobacterium tuberculosis,critical,5928,0.772,0.618,0.155


## 5 · Pathogen deep-dives
Geographic distribution and time trend for a critical pathogen–drug pair.

In [8]:
viz.resistance_choropleth('Acinetobacter baumannii', 'Meropenem')

In [9]:
viz.resistance_trend('Klebsiella pneumoniae', 'Meropenem')

## 6 · Surveillance blind spots & LMICs
Data volume by country. Blind spots (white) mark low-visibility regions — often LMICs where burden
may be high. The SPIDAAR real-world dataset (Kenya/Ghana/Uganda/Malawi) shows the point sharply.

In [10]:
viz.coverage_map()

In [11]:
long = pd.read_parquet(PROCESSED_DIR / "isolates_long.parquet",
                       columns=["source", "pathogen", "antibiotic", "sir", "country"])
sp = long[(long["source"] == "SPIDAAR") & (long["antibiotic"] == "Ceftriaxone")]
print("SPIDAAR (Sub-Saharan Africa) — % ceftriaxone-resistant (3GC-R):")
(sp.groupby("pathogen")["sir"].apply(lambda s: round((s == "R").mean()*100, 1))
   .rename("%R").reset_index())

SPIDAAR (Sub-Saharan Africa) — % ceftriaxone-resistant (3GC-R):


,pathogen,%R
0,Acinetobacter baumannii,75.0
1,Citrobacter freundii,100.0
2,Escherichia coli,82.0
3,Klebsiella aerogenes,100.0
4,Klebsiella oxytoca,100.0
5,Klebsiella pneumoniae,89.5
6,Providencia stuartii,100.0
7,Pseudomonas aeruginosa,90.9
8,Serratia marcescens,100.0


## 7 · Statistical rigor
We stress-test the framework so the conclusions are defensible, not arbitrary.

**(a) Resistance is rising — quantified.** Logistic regression of resistance on year gives an
odds ratio per year with a 95% confidence interval for each key pathogen–drug pair.

In [12]:
viz.trend_forest()

In [13]:
pd.read_parquet(PROCESSED_DIR / 'trend_models.parquet').round(4)

,pathogen,drug,n,OR_per_year,ci_lo,ci_hi,p_value
0,Acinetobacter baumannii,Meropenem,50712,1.0746,1.0707,1.0785,0.0000
1,Klebsiella pneumoniae,Meropenem,114295,1.1113,1.1065,1.1160,0.0000
2,Klebsiella pneumoniae,Ceftriaxone,57865,1.0219,1.0182,1.0256,0.0000
3,Escherichia coli,Ceftriaxone,78264,1.0235,1.0200,1.0271,0.0000
4,Escherichia coli,Ciprofloxacin,53117,1.0054,0.9981,1.0128,0.1485
5,Pseudomonas aeruginosa,Meropenem,126045,0.9787,0.9759,0.9815,0.0000
6,Enterococcus faecium,Vancomycin,21831,0.9500,0.9448,0.9552,0.0000
7,Staphylococcus aureus,Oxacillin,143036,0.8340,0.8313,0.8368,0.0000
8,Streptococcus pneumoniae,Penicillin,58705,0.9908,0.9868,0.9948,0.0000
9,Escherichia coli,Trimethoprim-sulfamethoxazole,60976,1.0126,1.0056,1.0197,0.0004


**(b) The index weighting is justified, not arbitrary.** Principal component analysis shows the
six indicators load on one dominant axis (PC1 explains ~72% of variance), and the data-driven
weights are close to equal — so equal weighting is defensible.

In [14]:
from src.models import pca_weights
pca = pca_weights()
print("PC1 explains %.0f%% of variance" % (pca['explained_variance'][0]*100))
pd.Series(pca['weights'], name='PCA-derived weight').round(3)

PC1 explains 72% of variance


prevalence    0.192
mic_drift     0.117
mdr           0.189
geo_spread    0.191
scarcity      0.162
pediatric     0.149
Name: PCA-derived weight, dtype: float64

**(c) Conclusions are robust to the weighting choice** (Spearman rank correlation vs equal weights).

In [15]:
pd.read_parquet(PROCESSED_DIR / 'sensitivity.parquet')

,scheme,spearman_vs_equal,top_pathogen
0,equal,1.000,Enterococcus faecium
1,prevalence_heavy,0.989,Acinetobacter baumannii
2,clinical,0.981,Enterococcus faecium
3,pca,0.990,Enterococcus faecium


**(d) Uncertainty is reported.** Bootstrap 95% confidence intervals for resistance prevalence
of key pathogen–drug pairs.

In [16]:
pd.read_parquet(PROCESSED_DIR / 'ci_keypairs.parquet').round(2)

,pathogen,drug,pctR,lo,hi,n
0,Acinetobacter baumannii,Meropenem,60.87,60.40,61.30,50712
1,Klebsiella pneumoniae,Meropenem,11.70,11.52,11.89,114295
2,Klebsiella pneumoniae,Ceftriaxone,31.53,31.16,31.93,57865
3,Escherichia coli,Ceftriaxone,21.71,21.41,22.00,78264
4,Escherichia coli,Ciprofloxacin,38.19,37.75,38.59,53117
5,Pseudomonas aeruginosa,Meropenem,20.24,20.02,20.45,126045
6,Enterococcus faecium,Vancomycin,26.18,25.61,26.76,21831
7,Staphylococcus aureus,Oxacillin,36.60,36.34,36.84,143036
8,Streptococcus pneumoniae,Penicillin,13.18,12.90,13.46,58705
9,Escherichia coli,Trimethoprim-sulfamethoxazole,40.44,40.08,40.83,60976


## 8 · Predicting surveillance blind spots (machine learning)
A gradient-boosted model predicts resistance from *generalizable* features (pathogen, drug, Gram,
continent, income tier, year) — never country identity — so it can estimate resistance for
countries/regions with **no surveillance**. We validate by holding out whole countries.

In [17]:
rs = pd.read_parquet(PROCESSED_DIR / 'rigor_summary.parquet').iloc[0]
print(f"Held-out-country cross-validation R2 : {rs['cv_R2_unseen_countries']:.2f}")
print(f"Weighted mean absolute error          : {rs['cv_weighted_MAE']*100:.1f}%")
print(f"Naive baseline (global mean) MAE      : {rs['baseline_MAE_global_mean']*100:.1f}%")

Held-out-country cross-validation R2 : 0.73
Weighted mean absolute error          : 8.1%
Naive baseline (global mean) MAE      : 18.4%


In [18]:
viz.blindspot_continent('Klebsiella pneumoniae', 'Meropenem')

## 9 · Key findings

- **The clearest treatment gaps are Gram-positive:** ***Enterococcus faecium* (VRE)** — high
  resistance need, little targeted R&D — and ***Staphylococcus epidermidis***. These are genuine
  gaps, not artifacts of how R&D is tagged.
- **Broad-spectrum Gram-negative R&D covers the Enterobacterales**, including the obscure ones
  (*Providencia*, *Proteus*, *Serratia*, *Citrobacter*). Attributing R&D only to *named* species
  would falsely flag these — our class-aware model corrects that.
- **WHO-critical headliners (*A. baumannii*, *K. pneumoniae*) and TB are well-served** — high need
  *and* high R&D attention. The system works where it focuses.
- **Resistance is actively worsening** for the critical Gram-negatives (e.g. *A. baumannii*
  meropenem, OR ≈ 1.07 per year, p < 0.001).
- **The biggest blind spot is geographic.** Sub-Saharan Africa is barely represented in surveillance,
  yet the SPIDAAR cohort shows **>80% third-generation-cephalosporin resistance** in *E. coli* and
  *K. pneumoniae*, and our model predicts high resistance across under-surveilled regions. High
  burden, low visibility — a double failure of surveillance *and* R&D reach.

### Stewardship implications
Prioritize (1) targeted R&D for VRE / Gram-positive gaps, (2) sustained investment in the
well-served critical Gram-negatives where resistance is still climbing, and (3) **expanding
surveillance in LMICs**, where the model flags the highest unmeasured resistance.

### Caveats (stated plainly)
- Only ATLAS ships native S/I/R; other datasets use a curated CLSI/TB breakpoint table
  (approximate, documented in `src/breakpoints.py`).
- R&D Hub records are tagged mostly at Gram-class level and by funder geography, so R&D attention
  is attributed at the Gram-class level (with a species-specific bonus); we do **not** over-claim
  species-level R&D differences.
- Income tiers/continents used in the prediction model are coarse, documented approximations.